# Citizen Assist AI — Modelagem Linear para Aprendizado de Máquina

**2º Global Solution 2026 | FIAP**  
**Tema:** perfil dos usuários de serviços públicos digitais no Brasil.

## Objetivo
Analisar dados reais da pesquisa **TIC Domicílios 2024 – Indivíduos**, do Cetic.br/NIC.br, para identificar padrões no uso de dois serviços públicos digitais (saúde pública e documentos pessoais), construir modelos de classificação e regressão e propor melhorias para uma solução Citizen Assist AI.


### Delimitação metodológica
O módulo G (Governo eletrônico) é aplicado a pessoas com **16 anos ou mais** que utilizaram a Internet nos últimos três meses. Por isso, as análises dos serviços G1 não representam automaticamente toda a população brasileira: representam os respondentes elegíveis com respostas válidas aos itens analisados. A pesquisa pergunta se a pessoa usou a Internet para procurar informações ou realizar serviços; portanto, o indicador não significa necessariamente que o serviço foi concluído integralmente online.

## 1. Preparação do ambiente
Vamos usar pandas e NumPy para organização, Matplotlib/Seaborn para gráficos, SciPy para testes estatísticos e scikit-learn para os modelos. O `python-docx` será usado para gerar o relatório final em Word.

In [1]:
!pip -q install pandas numpy matplotlib seaborn scipy scikit-learn statsmodels python-docx openpyxl

import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, mean_absolute_error,
    mean_squared_error, r2_score, roc_auc_score
)
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
PASTA_SAIDA = Path('/content/resultado_citizen_assist')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print('Ambiente preparado.')

Ambiente preparado.


## 2. Coleta da base pública
Fonte: [Cetic.br — Microdados TIC Domicílios 2024, Indivíduos](https://www.cetic.br/pt/arquivos/domicilios/2024/individuos/).

O portal informa que a pesquisa cobre o Brasil, possui unidade de análise indivíduos, é baseada em amostragem probabilística e inclui o Módulo G — Governo eletrônico. O arquivo CSV é baixado diretamente do endereço oficial abaixo.

In [5]:
URL_BASE = (

    'tic_domicilios_2024_individuos_base_de_microdados_v1.0.csv'
)
try:
    df = pd.read_csv(URL_BASE, sep=None, engine='python', encoding='latin-1')
    if df.shape[1] == 1:
        df = pd.read_csv(URL_BASE, sep=';', encoding='latin-1', low_memory=False)
except Exception as erro_download:
    print('Não foi possível baixar automaticamente:', erro_download)
    print('Baixe o CSV na página oficial do Cetic.br e selecione o arquivo nesta janela.')
    from google.colab import files
    arquivos_enviados = files.upload()
    csvs = [nome for nome in arquivos_enviados if nome.lower().endswith('.csv')]
    if not csvs:
        raise FileNotFoundError('Envie o arquivo CSV original da TIC Domicílios 2024 — Indivíduos.')
    caminho_csv = csvs[0]
    df = pd.read_csv(caminho_csv, sep=None, engine='python', encoding='latin-1')
    if df.shape[1] == 1:
        df = pd.read_csv(caminho_csv, sep=';', encoding='latin-1', low_memory=False)

print('Dimensões da base:', df.shape)
print('Primeiras colunas:', df.columns[:25].tolist())
display(df.head())

Dimensões da base: (21170, 439)
Primeiras colunas: ['QUEST', 'ID_DOMICILIO', 'ID_MORADOR', 'SEXO', 'IDADE', 'FAIXA_ETARIA', 'GRAU_INSTRUCAO_1', 'ESTUD', 'APOSENT', 'PEA', 'OCUP_12', 'OCUP_29', 'OCUP_32', 'RENDA_PESSOAL', 'RELIGIAO', 'RACA', 'C1', 'C2_D', 'C2_E', 'C2_F', 'C2_G', 'C2_I', 'C2_J', 'C2_OUTRO', 'C2A']


,QUEST,ID_DOMICILIO,ID_MORADOR,SEXO,IDADE,FAIXA_ETARIA,GRAU_INSTRUCAO_1,ESTUD,APOSENT,PEA,...,G1_AGREG,C8_NAO,I1A_NENHUM,J5_FAIXAS,J2_NENHUM,J3_AGREG,J3AB,C5_COB_AGREG,C5_COB_NENHUM,C6_COB_NENHUM
0,200809622,1,3,2,83,6,1,0,1,5,...,0,1,1,1,99,1,2,0,1,1
1,200809621,2,1,1,63,6,1,0,1,1,...,0,1,1,0,1,0,4,0,1,1
2,200820881,4,1,1,70,6,10,0,1,1,...,1,1,1,1,99,1,2,0,1,1
3,200818341,6,1,1,85,6,1,0,0,5,...,0,1,1,0,1,0,4,0,1,1
4,200818340,7,1,2,54,5,15,0,0,1,...,0,1,1,1,99,1,2,0,1,1


### 2.1 Conferência dos nomes das variáveis
O questionário identifica os serviços do Módulo G como `G1_A` a `G1_G`. As variáveis sociodemográficas podem ter rótulos abreviados no arquivo de microdados. A célula seguinte normaliza nomes e procura os campos comuns. **Confira a lista impressa**; se algum campo sociodemográfico não for localizado, ajuste apenas a lista de apelidos na célula abaixo. Os dois campos de serviço `G1_A` e `G1_B` são obrigatórios.

In [ ]:
def normalizar_nome(valor):
    texto = unicodedata.normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode('ascii')
    texto = texto.upper().strip()
    texto = re.sub(r'[^A-Z0-9]+', '_', texto)
    return texto.strip('_')

df.columns = [normalizar_nome(c) for c in df.columns]
print('Colunas disponíveis na base:')
print(df.columns.tolist())

# Variáveis dos dois serviços principais e dos sete serviços do questionário.
SERVICOS = {
    'G1_A': 'Documentos pessoais',
    'G1_B': 'Saúde pública',
    'G1_C': 'Educação pública',
    'G1_D': 'Direitos do trabalhador / previdência',
    'G1_E': 'Impostos e taxas',
    'G1_F': 'Polícia e segurança',
    'G1_G': 'Transporte e serviços urbanos',
}
servicos_presentes = [c for c in SERVICOS if c in df.columns]
if not {'G1_A', 'G1_B'}.issubset(df.columns):
    raise KeyError('Não encontrei G1_A e G1_B. Confira os nomes impressos acima e ajuste o dicionário de variáveis.')

# Ajuste os apelidos abaixo somente se a busca automática não encontrar uma variável.
APELIDOS = {
    'sexo': ['SEXO', 'P_SEXO', 'R_SEXO'],
    'idade': ['IDADE', 'IDADE_EXATA', 'IDADE_ANOS', 'P_IDADE'],
    'faixa_etaria': ['FAIXA_ETARIA', 'IDADE_FAIXA', 'FAIXA_DE_IDADE'],
    'escolaridade': ['ESCOLARIDADE', 'P_ESCOLARIDADE', 'GRAU_INSTRUCAO', 'P_GRAU_INSTRUCAO', 'GRAU_DE_INSTRUCAO', 'INSTRUCAO', 'EDUCACAO'],
    'renda': ['RENDA_FAMILIAR', 'FAIXA_RENDA_FAMILIAR', 'RENDAFAM', 'FAIXA_RENDA', 'RENDA_FAM', 'P_RENDA', 'RENDA'],
    'classe': ['CLASSE', 'P_CLASSE', 'CLASSE_SOCIAL', 'CLASSE_ECONOMICA', 'CLASS'],
}

def encontrar_coluna(apelidos, colunas):
    for candidato in apelidos:
        c = normalizar_nome(candidato)
        if c in colunas:
            return c
    return None

MAPA = {nome: encontrar_coluna(lista, df.columns) for nome, lista in APELIDOS.items()}
print('Mapeamento sociodemográfico localizado:')
print(MAPA)
print('Serviços identificados:', servicos_presentes)

## 3. Limpeza e preparação dos dados
No questionário, o código `1` representa “Sim” e `2` representa “Não”. Os códigos `8` e `9` indicam “Não sabe” e “Não respondeu”; eles serão tratados como ausentes, nunca como “Não”. Como o Módulo G tem filtro de idade e uso recente da Internet, o conjunto analítico será formado por respondentes com respostas válidas (`1` ou `2`) nos dois serviços principais.

O índice de habilidades digitais será calculado a partir dos itens `I1A_A` a `I1A_L`, quando disponíveis: proporção de respostas “Sim” entre os itens respondidos. Esse índice é uma variável explicativa auxiliar, não uma escala oficial do Cetic.br.

In [ ]:
# Converte respostas de serviços para números, sem transformar 8/9 em não uso.
for col in servicos_presentes:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[~df[col].isin([1, 2]), col] = np.nan

# Cria a amostra analítica com respostas válidas para documentos e saúde.
base = df[df[['G1_A', 'G1_B']].notna().all(axis=1)].copy()
base['uso_documentos'] = (base['G1_A'] == 1).astype(int)
base['uso_saude'] = (base['G1_B'] == 1).astype(int)

# Índice de habilidades digitais, se os itens correspondentes estiverem na base.
itens_habilidades = [f'I1A_{letra}' for letra in list('ABCDEFGHIJKL') if f'I1A_{letra}' in df.columns]
if len(itens_habilidades) >= 4:
    for col in itens_habilidades:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df.loc[~df[col].isin([1, 2]), col] = np.nan
    base = df.loc[base.index].copy()
    respostas_hab = base[itens_habilidades].replace({1: 1, 2: 0})
    base['indice_habilidades_digitais'] = respostas_hab.mean(axis=1)
else:
    base['indice_habilidades_digitais'] = np.nan

# Recria os alvos após o recorte e registra o tamanho da amostra.
base['uso_documentos'] = (base['G1_A'] == 1).astype(int)
base['uso_saude'] = (base['G1_B'] == 1).astype(int)
print(f'Registros brutos: {len(df):,}')
print(f'Registros válidos para os dois serviços: {len(base):,}')
print(f'Itens de habilidades digitais encontrados: {len(itens_habilidades)}')
print('Proporção de respostas válidas nos dois serviços:', round(len(base) / len(df) * 100, 2), '%')

### 3.1 Construção das variáveis explicativas
Idade exata é tratada como numérica; faixa etária, sexo, escolaridade, renda e classe são tratados como categorias. A regressão não interpreta códigos categóricos como se fossem distâncias numéricas: as categorias passam por codificação *one-hot*.

In [ ]:
features = pd.DataFrame(index=base.index)
colunas_numericas = []
colunas_categoricas = []

if MAPA['idade']:
    features['idade'] = pd.to_numeric(base[MAPA['idade']], errors='coerce')
    colunas_numericas.append('idade')
elif MAPA['faixa_etaria']:
    features['faixa_etaria'] = base[MAPA['faixa_etaria']].astype('string')
    colunas_categoricas.append('faixa_etaria')

for variavel in ['sexo', 'escolaridade', 'renda', 'classe']:
    coluna = MAPA[variavel]
    if coluna:
        nome = variavel
        features[nome] = base[coluna].astype('string').replace({'nan': pd.NA, 'None': pd.NA})
        colunas_categoricas.append(nome)

if base['indice_habilidades_digitais'].notna().sum() > 0:
    features['indice_habilidades_digitais'] = base['indice_habilidades_digitais']
    colunas_numericas.append('indice_habilidades_digitais')

# Remove atributos sem informação ou com valor único.
remover = [c for c in features.columns if features[c].notna().sum() == 0 or features[c].nunique(dropna=True) <= 1]
features = features.drop(columns=remover)
colunas_numericas = [c for c in colunas_numericas if c in features.columns]
colunas_categoricas = [c for c in colunas_categoricas if c in features.columns]

if features.shape[1] == 0:
    raise ValueError('Nenhuma variável explicativa foi identificada. Revise o mapeamento sociodemográfico.')

print('Variáveis explicativas:', features.columns.tolist())
print('Numéricas:', colunas_numericas)
print('Categóricas:', colunas_categoricas)
display(features.head())

## 4. Análise exploratória e estatística descritiva
Vamos observar o perfil da amostra, a frequência de uso dos serviços, valores ausentes e distribuição das variáveis explicativas.

In [ ]:
# Tabela de prevalência dos serviços (percentual dentro da amostra válida de cada item).
linhas_servicos = []
for col, rotulo in SERVICOS.items():
    if col in base.columns:
        validos = base[col].dropna()
        linhas_servicos.append({
            'Variável': col,
            'Serviço': rotulo,
            'Respostas válidas': int(len(validos)),
            'Usaram (n)': int((validos == 1).sum()),
            'Não usaram (n)': int((validos == 2).sum()),
            'Uso (%)': float((validos == 1).mean() * 100) if len(validos) else np.nan,
        })
prevalencia = pd.DataFrame(linhas_servicos).sort_values('Uso (%)', ascending=False)
display(prevalencia.round(2))
prevalencia.to_csv(PASTA_SAIDA / 'prevalencia_servicos.csv', index=False, encoding='utf-8-sig')

# Estatística descritiva das variáveis numéricas.
if colunas_numericas:
    display(features[colunas_numericas].describe().T)

# Frequências das variáveis categóricas disponíveis.
for col in colunas_categoricas:
    print(f'\nDistribuição de {col}:')
    display(features[col].value_counts(dropna=False).rename_axis(col).to_frame('Quantidade'))

# Verificação de dados ausentes na amostra analítica.
missing = (features.isna().mean() * 100).sort_values(ascending=False).rename('Ausentes (%)').to_frame()
display(missing.round(2))

### 4.1 Visualização da frequência dos serviços

In [ ]:
plt.figure(figsize=(10, 5))
ordem = prevalencia.sort_values('Uso (%)', ascending=True)
plt.barh(ordem['Serviço'], ordem['Uso (%)'])
plt.xlabel('Respondentes que usaram o serviço (%)')
plt.ylabel('Tipo de serviço público')
plt.title('Uso de serviços públicos pela Internet — amostra analítica')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'uso_servicos.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Probabilidades e análise bivariada
Serão calculadas probabilidades condicionais empíricas, por exemplo: proporção de uso de saúde pública entre respondentes de cada categoria de escolaridade. Em seguida, serão avaliadas associações entre o uso do serviço e as variáveis categóricas pelo teste qui-quadrado de independência. Para a idade numérica, será usado o teste de Mann–Whitney entre grupos de uso e não uso.

**Hipóteses do qui-quadrado:**  
H₀: a variável sociodemográfica e o uso do serviço são independentes.  
H₁: existe associação entre a variável sociodemográfica e o uso do serviço.

In [ ]:
def qui_quadrado_categorico(dados, variavel, alvo, nome_alvo):
    tabela = pd.crosstab(dados[variavel], dados[alvo])
    if tabela.shape[0] < 2 or tabela.shape[1] < 2:
        return None
    chi2, p, gl, esperados = chi2_contingency(tabela)
    n = tabela.to_numpy().sum()
    minimo = min(tabela.shape)
    cramers_v = np.sqrt(chi2 / (n * (minimo - 1))) if minimo > 1 and n > 0 else np.nan
    return {
        'Variável': variavel, 'Serviço': nome_alvo, 'Qui-quadrado': chi2,
        'Graus de liberdade': gl, 'p-valor': p, 'V de Cramer': cramers_v,
        'Menor frequência esperada': float(np.min(esperados))
    }

resultados_testes = []
for variavel in colunas_categoricas:
    for alvo, nome in [('uso_saude', 'Saúde pública'), ('uso_documentos', 'Documentos pessoais')]:
        res = qui_quadrado_categorico(base.join(features[[variavel]]), variavel, alvo, nome)
        if res:
            resultados_testes.append(res)

testes_df = pd.DataFrame(resultados_testes)
if not testes_df.empty:
    testes_df['Conclusão (5%)'] = np.where(testes_df['p-valor'] < 0.05, 'Rejeita H0; associação estatística', 'Não rejeita H0')
    display(testes_df.round(4))
    testes_df.to_csv(PASTA_SAIDA / 'testes_hipotese.csv', index=False, encoding='utf-8-sig')
else:
    print('Não foi possível realizar o qui-quadrado com as variáveis categóricas disponíveis.')

# Probabilidade condicional por escolaridade ou sexo, conforme disponibilidade.
variavel_grupo = 'escolaridade' if 'escolaridade' in features.columns else (colunas_categoricas[0] if colunas_categoricas else None)
if variavel_grupo:
    probabilidade = base.join(features[[variavel_grupo]]).groupby(variavel_grupo, dropna=False).agg(
        P_uso_saude=('uso_saude', 'mean'),
        P_uso_documentos=('uso_documentos', 'mean'),
        N=('uso_saude', 'size')
    ).reset_index()
    probabilidade['P_uso_saude_%'] = probabilidade['P_uso_saude'] * 100
    probabilidade['P_uso_documentos_%'] = probabilidade['P_uso_documentos'] * 100
    display(probabilidade.round(3))
    probabilidade.to_csv(PASTA_SAIDA / 'probabilidades_condicionais.csv', index=False, encoding='utf-8-sig')

# Comparação de idade entre usuários e não usuários, se idade exata estiver disponível.
if 'idade' in features.columns:
    idade_df = base[['uso_saude', 'uso_documentos']].join(features[['idade']]).dropna(subset=['idade'])
    for alvo, nome in [('uso_saude', 'Saúde pública'), ('uso_documentos', 'Documentos pessoais')]:
        g1 = idade_df.loc[idade_df[alvo] == 1, 'idade']
        g0 = idade_df.loc[idade_df[alvo] == 0, 'idade']
        if len(g1) >= 5 and len(g0) >= 5:
            u, p = mannwhitneyu(g1, g0, alternative='two-sided')
            print(f'Mann–Whitney — {nome}: mediana usuários={g1.median():.1f}; não usuários={g0.median():.1f}; p={p:.5f}')

## 6. Análise multivariada — regressão logística
Serão ajustados dois classificadores binários:
- **Modelo 1:** estimar se o respondente usou serviços digitais de saúde pública (`G1_B`).
- **Modelo 2:** estimar se o respondente usou serviços digitais de documentos pessoais (`G1_A`).

As variáveis explicativas serão os atributos sociodemográficos localizados e o índice de habilidades digitais, quando disponível. Os dados serão separados em treino (80%) e teste (20%). A avaliação utilizará acurácia, precisão, revocação, F1 e matriz de confusão. A interpretação deve considerar que a classificação estima associação/predição e não prova causalidade.

In [ ]:
def montar_preprocessador(numericas, categoricas):
    transformadores = []
    if numericas:
        pipeline_num = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformadores.append(('num', pipeline_num, numericas))
    if categoricas:
        pipeline_cat = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])
        transformadores.append(('cat', pipeline_cat, categoricas))
    return ColumnTransformer(transformadores, remainder='drop')

resultados_logisticos = {}
metricas_classificacao = []
matrizes_confusao = {}

for alvo, nome in [('uso_saude', 'Saúde pública'), ('uso_documentos', 'Documentos pessoais')]:
    dados_modelo = base.join(features)
    X = dados_modelo[features.columns]
    y = dados_modelo[alvo].astype(int)
    if y.nunique() < 2 or y.value_counts().min() < 5 or len(X) < 50:
        print(f'Modelo de {nome} não ajustado: amostra/classe insuficiente.')
        continue

    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    pipeline = Pipeline([
        ('preprocessamento', montar_preprocessador(colunas_numericas, colunas_categoricas)),
        ('modelo', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
    ])
    pipeline.fit(X_treino, y_treino)
    pred = pipeline.predict(X_teste)
    proba = pipeline.predict_proba(X_teste)[:, 1]

    m = {
        'Serviço': nome,
        'N teste': len(y_teste),
        'Acurácia': accuracy_score(y_teste, pred),
        'Precisão': precision_score(y_teste, pred, zero_division=0),
        'Revocação': recall_score(y_teste, pred, zero_division=0),
        'F1': f1_score(y_teste, pred, zero_division=0),
        'AUC': roc_auc_score(y_teste, proba) if y_teste.nunique() == 2 else np.nan,
    }
    metricas_classificacao.append(m)
    matrizes_confusao[nome] = confusion_matrix(y_teste, pred, labels=[0, 1])
    resultados_logisticos[nome] = pipeline
    print(f'\nRelatório de classificação — {nome}')
    print(classification_report(y_teste, pred, zero_division=0))
    print('Matriz de confusão [ [TN, FP], [FN, TP] ]:')
    print(matrizes_confusao[nome])

metricas_logit_df = pd.DataFrame(metricas_classificacao)
if not metricas_logit_df.empty:
    display(metricas_logit_df.round(4))
    metricas_logit_df.to_csv(PASTA_SAIDA / 'metricas_regressao_logistica.csv', index=False, encoding='utf-8-sig')

### 6.1 Matrizes de confusão
Na matriz, a primeira linha representa os casos reais sem uso do serviço (TN e FP); a segunda representa os casos reais com uso (FN e TP). Falsos negativos merecem atenção em um assistente público, pois representam cidadãos cujo uso do serviço não foi identificado pelo modelo.

In [ ]:
for nome, matriz in matrizes_confusao.items():
    plt.figure(figsize=(4.5, 3.5))
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predito: não', 'Predito: sim'],
                yticklabels=['Real: não', 'Real: sim'])
    plt.title(f'Matriz de confusão — {nome}')
    plt.tight_layout()
    nome_arquivo = 'matriz_' + normalizar_nome(nome).lower() + '.png'
    plt.savefig(PASTA_SAIDA / nome_arquivo, dpi=180, bbox_inches='tight')
    plt.show()

## 7. Modelagem linear — regressão linear múltipla
Para integrar os diferentes tipos de atendimento em um indicador único, será calculada a **quantidade de tipos de serviço público** utilizados por cada respondente, de 0 a 7. A regressão linear tentará estimar esse total com as variáveis sociodemográficas e o índice de habilidades digitais disponíveis.

Como o alvo é uma contagem limitada entre 0 e 7, o modelo linear é usado aqui como exercício didático de modelagem linear, conforme a disciplina; não é o único modelo estatístico possível para esse tipo de variável. Serão avaliados R², MAE e RMSE.

In [ ]:
colunas_g1_completas = [c for c in SERVICOS if c in base.columns]
base_linear = base[base[colunas_g1_completas].notna().all(axis=1)].copy()
base_linear['quantidade_tipos_servico'] = (base_linear[colunas_g1_completas] == 1).sum(axis=1)
X_linear = features.loc[base_linear.index]
y_linear = base_linear['quantidade_tipos_servico'].astype(float)

metricas_linear = None
modelo_linear = None
if len(base_linear) >= 50 and y_linear.nunique() > 1:
    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X_linear, y_linear, test_size=0.20, random_state=RANDOM_STATE
    )
    modelo_linear = Pipeline([
        ('preprocessamento', montar_preprocessador(colunas_numericas, colunas_categoricas)),
        ('modelo', LinearRegression())
    ])
    modelo_linear.fit(X_treino, y_treino)
    pred_linear = modelo_linear.predict(X_teste)
    metricas_linear = {
        'N teste': len(y_teste),
        'R²': r2_score(y_teste, pred_linear),
        'MAE': mean_absolute_error(y_teste, pred_linear),
        'RMSE': mean_squared_error(y_teste, pred_linear) ** 0.5,
        'Média real': y_teste.mean(),
        'Média prevista': pred_linear.mean(),
    }
    display(pd.DataFrame([metricas_linear]).round(4))
    pd.DataFrame([metricas_linear]).to_csv(PASTA_SAIDA / 'metricas_regressao_linear.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(6, 5))
    plt.scatter(y_teste, pred_linear, alpha=0.5)
    plt.plot([0, 7], [0, 7], linestyle='--')
    plt.xlabel('Quantidade real de tipos de serviço')
    plt.ylabel('Quantidade estimada')
    plt.title('Regressão linear — valores reais x estimados')
    plt.tight_layout()
    plt.savefig(PASTA_SAIDA / 'regressao_linear_real_vs_previsto.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('Regressão linear não ajustada: quantidade insuficiente de registros completos ou variação no alvo.')

## 8. Interpretação dos resultados e aplicações Citizen Assist AI
Use os números impressos nas células anteriores para responder às perguntas:
1. Quais dos dois serviços (saúde ou documentos) aparece com maior proporção de uso na amostra?
2. As proporções variam entre categorias de escolaridade, sexo, classe ou renda?
3. Quais associações foram estatisticamente significativas (p < 0,05)? Qual o tamanho do efeito (V de Cramer)?
4. O modelo logístico teve melhor desempenho para qual serviço? Observe precisão e revocação, não apenas acurácia.
5. A regressão linear conseguiu explicar parte da variação na quantidade de serviços utilizados? Interprete o R² e os erros.
6. Que funcionalidades do Citizen Assist AI podem reduzir barreiras observadas: linguagem simples, atendimento por texto/voz, orientação passo a passo, roteamento para órgãos e encaminhamento humano?

**Cuidado:** os resultados são associativos e preditivos, não causais. O módulo G mede busca de informação ou realização de serviço na Internet e não mede, por si só, satisfação com o atendimento.

## 9. Exportação do relatório final
A última célula cria um arquivo Word usando as métricas e tabelas calculadas nesta execução. Revise o documento, complete os nomes dos integrantes e confira se as saídas e gráficos correspondem ao que foi obtido no notebook antes de entregar.

In [ ]:
# Gera um relatório dinâmico a partir dos resultados calculados nas células anteriores.
from docx.shared import RGBColor

RELATORIO = PASTA_SAIDA / 'Relatorio_Final_Citizen_Assist_AI.docx'

def adicionar_tabela(doc, dataframe, max_linhas=20):
    if dataframe is None or len(dataframe) == 0:
        doc.add_paragraph('Não há resultados disponíveis para esta tabela.')
        return
    data = dataframe.head(max_linhas).copy()
    tabela = doc.add_table(rows=1, cols=len(data.columns))
    tabela.style = 'Table Grid'
    tabela.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, col in enumerate(data.columns):
        tabela.rows[0].cells[i].text = str(col)
    for _, linha in data.iterrows():
        celulas = tabela.add_row().cells
        for i, valor in enumerate(linha):
            if isinstance(valor, (float, np.floating)):
                celulas[i].text = f'{valor:.3f}'
            else:
                celulas[i].text = str(valor)

def porcentagem(valor):
    return f'{valor:.1f}%'

doc = Document()
sec = doc.sections[0]
sec.top_margin = Inches(0.75)
sec.bottom_margin = Inches(0.75)
styles = doc.styles
styles['Normal'].font.name = 'Arial'
styles['Normal'].font.size = Pt(10)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run('CITIZEN ASSIST AI')
r.bold = True; r.font.size = Pt(20); r.font.color.rgb = RGBColor(31, 78, 121)
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run('MODELAGEM LINEAR PARA APRENDIZADO DE MÁQUINA')
r.bold = True; r.font.size = Pt(14)
p = doc.add_paragraph('1º Global Solution 2026 | FIAP')
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p = doc.add_paragraph('Integrantes: ______________________________________________')
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p = doc.add_paragraph('Turma: ____________________    Data: ____/____/______')
p.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_page_break()
doc.add_heading('1. Introdução', level=1)
doc.add_paragraph(
    'A digitalização dos serviços públicos pode ampliar o acesso à informação e reduzir deslocamentos, '
    'mas seus benefícios dependem de conectividade, habilidades digitais e facilidade de uso. Identificar '
    'os perfis associados ao uso de serviços públicos digitais permite direcionar comunicação, suporte e '
    'interfaces para diferentes necessidades dos cidadãos. Este estudo se relaciona ao desafio Citizen Assist AI, '
    'que propõe um assistente de inteligência artificial para facilitar o acesso a informações e serviços públicos.'
)
doc.add_paragraph(
    'O estudo utiliza a Pesquisa TIC Domicílios 2024, do Cetic.br/NIC.br, e compara dois grupos de serviços: '
    'saúde pública (G1_B) e documentos pessoais (G1_A). A publicação oficial da pesquisa informa, como referência '
    'nacional ponderada, que 61% dos usuários de Internet com 16 anos ou mais utilizaram algum serviço de governo '
    'eletrônico em 2024; saúde pública foi mencionada por 32% e documentos pessoais por 31%. Esses números são '
    'referências publicadas pelo Cetic.br e não devem ser confundidos com os resultados dos modelos deste trabalho.'
)

doc.add_heading('2. Metodologia', level=1)
doc.add_paragraph(
    'Foi selecionada a base pública TIC Domicílios 2024 – Indivíduos, disponibilizada em CSV pelo Cetic.br. '
    'A pesquisa possui abrangência nacional, unidade de análise indivíduos e desenho de amostragem probabilística. '
    'O Módulo G é aplicado a pessoas com 16 anos ou mais que utilizaram Internet nos últimos três meses. '
    'A análise foi restrita aos registros com respostas válidas (1 = sim; 2 = não) nos dois serviços principais. '
    'Respostas 8 (não sabe), 9 (não respondeu) e valores ausentes foram tratados como ausentes, não como ausência de uso.'
)
doc.add_paragraph(
    'As variáveis explicativas utilizadas foram as características sociodemográficas disponíveis na base '
    '(sexo, idade ou faixa etária, escolaridade, renda familiar e classe econômica) e, quando disponível, '
    'um índice exploratório de habilidades digitais derivado dos itens I1A_A a I1A_L. Variáveis categóricas '
    'foram codificadas por one-hot encoding; valores ausentes foram imputados pela mediana (numéricas) ou moda (categóricas).'
)
doc.add_paragraph(
    'A análise exploratória incluiu estatísticas descritivas e frequências dos serviços. A análise bivariada incluiu '
    'probabilidades condicionais, teste qui-quadrado de independência e, quando havia idade exata, teste de Mann–Whitney. '
    'A análise multivariada utilizou regressão logística para classificar o uso dos serviços de saúde e de documentos. '
    'A regressão linear múltipla estimou a quantidade de tipos de serviço utilizados, entre zero e sete. '
    'A avaliação dos modelos utilizou separação treino/teste de 80%/20%, com semente aleatória fixa (42).'
)

doc.add_heading('3. Resultados', level=1)
doc.add_paragraph(f'A base original contém {len(df):,} registros. Após selecionar respostas válidas em G1_A e G1_B, a amostra analítica contém {len(base):,} registros.'.replace(',', '.'))
doc.add_heading('3.1 Frequência dos serviços', level=2)
add = prevalencia[['Serviço', 'Respostas válidas', 'Usaram (n)', 'Uso (%)']].copy()
add['Uso (%)'] = add['Uso (%)'].map(lambda x: f'{x:.1f}%')
adicionar_tabela(doc, add)
fig = PASTA_SAIDA / 'uso_servicos.png'
if fig.exists(): doc.add_picture(str(fig), width=Inches(6.0))

doc.add_heading('3.2 Testes de hipótese e probabilidades', level=2)
if not testes_df.empty:
    mostrar = testes_df[['Variável', 'Serviço', 'Qui-quadrado', 'Graus de liberdade', 'p-valor', 'V de Cramer', 'Conclusão (5%)']]
    adicionar_tabela(doc, mostrar.round(4))
else:
    doc.add_paragraph('Não houve tabela de testes disponível nesta execução.')
if 'probabilidade' in globals():
    doc.add_paragraph('Probabilidades condicionais estimadas por categoria:')
    col_prob = [c for c in probabilidade.columns if c in [variavel_grupo, 'N', 'P_uso_saude_%', 'P_uso_documentos_%']]
    adicionar_tabela(doc, probabilidade[col_prob].round(2))

doc.add_heading('3.3 Regressão logística', level=2)
if not metricas_logit_df.empty:
    adicionar_tabela(doc, metricas_logit_df.round(4))
    for nome, matriz in matrizes_confusao.items():
        doc.add_paragraph(f'Matriz de confusão — {nome} ([TN, FP]; [FN, TP]): {matriz.tolist()}')
        nome_arquivo = 'matriz_' + normalizar_nome(nome).lower() + '.png'
        caminho = PASTA_SAIDA / nome_arquivo
        if caminho.exists(): doc.add_picture(str(caminho), width=Inches(4.5))
else:
    doc.add_paragraph('Os classificadores logísticos não puderam ser ajustados nesta execução.')

doc.add_heading('3.4 Regressão linear', level=2)
if metricas_linear:
    adicionar_tabela(doc, pd.DataFrame([metricas_linear]).round(4))
    caminho = PASTA_SAIDA / 'regressao_linear_real_vs_previsto.png'
    if caminho.exists(): doc.add_picture(str(caminho), width=Inches(5.2))
else:
    doc.add_paragraph('A regressão linear não pôde ser ajustada nesta execução por insuficiência de registros completos ou variação no alvo.')

doc.add_heading('4. Análise e discussão', level=1)
if len(prevalencia) >= 2:
    top = prevalencia.iloc[0]
    segundo = prevalencia.iloc[1]
    doc.add_paragraph(
        f"Na amostra analisada, o serviço com maior proporção de uso foi {top['Serviço']} "
        f"({top['Uso (%)']:.1f}%), seguido por {segundo['Serviço']} ({segundo['Uso (%)']:.1f}%). "
        'A comparação descreve os respondentes com respostas válidas e não substitui estimativas ponderadas do levantamento.'
    )
if not testes_df.empty:
    significativos = testes_df[testes_df['p-valor'] < 0.05]
    doc.add_paragraph(
        f"Foram observadas {len(significativos)} associações estatisticamente significativas (p < 0,05) "
        f"entre as combinações de variáveis avaliadas ({len(testes_df)} testes no total). "
        'O V de Cramer deve ser consultado para avaliar a intensidade da associação, pois significância estatística não representa necessariamente efeito forte.'
    )
if not metricas_logit_df.empty:
    melhor = metricas_logit_df.sort_values('F1', ascending=False).iloc[0]
    doc.add_paragraph(
        f"Entre os classificadores avaliados, {melhor['Serviço']} apresentou o maior F1 ({melhor['F1']:.3f}). "
        'Precisão indica a proporção de previsões positivas corretas; revocação indica a proporção de usuários reais identificados. '
        'Em um assistente público, a revocação pode ser particularmente importante para reduzir a chance de deixar de reconhecer cidadãos que necessitam de orientação.'
    )
if metricas_linear:
    doc.add_paragraph(
        f"A regressão linear apresentou R² = {metricas_linear['R²']:.3f}, MAE = {metricas_linear['MAE']:.3f} "
        f"e RMSE = {metricas_linear['RMSE']:.3f}. O R² representa a fração da variação do total de tipos de serviços explicada pelo modelo na amostra de teste; "
        'valores de erro devem ser interpretados na escala de zero a sete serviços.'
    )
doc.add_paragraph(
    'Os achados podem orientar o Citizen Assist AI a priorizar linguagem simples, fluxos guiados para saúde e documentos, '
    'busca inteligente de serviços, atendimento por texto e voz e encaminhamento humano quando a demanda exigir análise especializada. '
    'Perfis com menor probabilidade estimada de uso devem receber suporte inclusivo, sem restrição de acesso ou tratamento discriminatório.'
)

doc.add_heading('5. Conclusões', level=1)
doc.add_paragraph(
    'A análise permite caracterizar padrões de utilização de serviços públicos digitais e testar se variáveis sociodemográficas '
    'e habilidades digitais ajudam a estimar o uso de saúde pública, documentos pessoais e a quantidade de tipos de serviço utilizados. '
    'Os modelos devem ser usados como instrumentos de planejamento e melhoria de atendimento, e não como mecanismos para negar serviços ou '
    'presumir necessidades individuais. Recomenda-se ampliar a validação com desenho amostral complexo, dados longitudinais e avaliações reais de usabilidade.'
)

doc.add_heading('Limitações', level=2)
doc.add_paragraph(
    'A análise apresentada é baseada em respostas autorrelatadas; o módulo G é filtrado para usuários de Internet de 16 anos ou mais; '
    'o indicador G1 combina busca de informações e realização de serviços; e a execução didática dos testes/modelos não incorpora, por padrão, '
    'todos os estratos, conglomerados e pesos do plano amostral. Portanto, os resultados computados no notebook não devem ser interpretados como estimativas oficiais nacionais. '
    'O índice de habilidades digitais é uma construção exploratória para este trabalho.'
)

doc.add_heading('Referências', level=1)
doc.add_paragraph('NÚCLEO DE INFORMAÇÃO E COORDENAÇÃO DO PONTO BR (NIC.br). Pesquisa sobre o uso das tecnologias de informação e comunicação nos domicílios brasileiros: Pesquisa TIC Domicílios, ano 2024. São Paulo: CGI.br, 2025. Disponível em: https://www.nic.br/media/docs/publicacoes/2/20250512120132/tic_domicilios_2024_livro_eletronico.pdf')
doc.add_paragraph('CETIC.br. Microdados TIC Domicílios 2024 — Indivíduos. Disponível em: https://www.cetic.br/pt/arquivos/domicilios/2024/individuos/')
doc.add_paragraph('CETIC.br. Questionário TIC Domicílios 2024. Módulo G — Governo eletrônico. Disponível em: https://cetic.br/media/microdados/855/tic_domicilios_2024_questionario_v1.0.pdf')

doc.save(RELATORIO)
print('Relatório final gerado em:', RELATORIO)
print('Pasta com resultados e gráficos:', PASTA_SAIDA)

## 10. Preparação da entrega em ZIP
O arquivo ZIP deve conter o relatório final gerado, o notebook `.ipynb` e, de preferência, as tabelas e gráficos exportados. No Colab, faça **Arquivo → Fazer download → Fazer download do .ipynb** para baixar o notebook. Depois, use a célula abaixo para enviar esse `.ipynb` e incluí-lo junto do relatório na pasta compactada.

In [ ]:
# Execute esta célula depois de baixar o notebook .ipynb pelo menu Arquivo.
# Selecione o .ipynb baixado quando a janela de upload aparecer.
from google.colab import files
import zipfile

print('Selecione o arquivo .ipynb deste projeto para adicioná-lo ao ZIP final.')
uploaded = files.upload()
notebooks = [nome for nome in uploaded if nome.lower().endswith('.ipynb')]

zip_final = Path('/content/Entrega_Citizen_Assist_AI.zip')
with zipfile.ZipFile(zip_final, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(RELATORIO, arcname='Relatorio_Final_Citizen_Assist_AI.docx')
    for arquivo in PASTA_SAIDA.glob('*'):
        if arquivo.is_file() and arquivo.name != RELATORIO.name:
            z.write(arquivo, arcname=f'resultados/{arquivo.name}')
    if notebooks:
        z.write(notebooks[0], arcname='Notebook_Citizen_Assist_AI.ipynb')
    else:
        print('ATENÇÃO: nenhum .ipynb foi enviado. O ZIP ficará sem o código-fonte; repita esta etapa após baixar o notebook.')

print('ZIP criado:', zip_final)
files.download(str(zip_final))